# 21 — GNN-SDM Production Training

Train GNN-SDM for all qualifying species using the best architecture
from notebook 20. Compare results with the RF baseline.

**Architecture**: set `best_hidden_dims` based on notebook 20 results.

### Setup and load graph

In [1]:
import config
import numpy as np
import pandas as pd
import pickle
import json
import time
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import networkx as nx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Load graph
with open('landscape_graph.pkl', 'rb') as f:
    G = pickle.load(f)

# Load patch features and names
patch_features = np.load('patch_features.npy')
with open('patch_feature_names.json') as f:
    feature_names = json.load(f)

n_patches = G.number_of_nodes()
n_features = len(feature_names)
print(f'Graph: {n_patches:,} nodes, {G.number_of_edges():,} edges, '
      f'{n_features} features')


/home/sagemaker-user/.conda/envs/cas/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: Tesla T4
Graph: 166,464 nodes, 465,833 edges, 12 features


### Convert to PyG Data object

In [2]:
# Node features
X = torch.tensor(patch_features, dtype=torch.float32)

# Normalize features (zero mean, unit variance)
X_mean = X.mean(dim=0)
X_std = X.std(dim=0).clamp(min=1e-6)
X = (X - X_mean) / X_std

# Edge index (PyG format: 2 × n_edges, both directions for undirected)
edges = list(G.edges())
src = [e[0] for e in edges] + [e[1] for e in edges]
dst = [e[1] for e in edges] + [e[0] for e in edges]
edge_index = torch.tensor([src, dst], dtype=torch.long)

# PageRank for presence weighting
print('Computing PageRank...')
pagerank = nx.pagerank(G)
pr_values = torch.tensor([pagerank[i] for i in range(n_patches)], dtype=torch.float32)

print(f'Node features: {X.shape}')
print(f'Edge index: {edge_index.shape}')
print(f'PageRank: min={pr_values.min():.6f}, max={pr_values.max():.6f}')


Computing PageRank...
Node features: torch.Size([166464, 12])
Edge index: torch.Size([2, 931666])
PageRank: min=0.000002, max=0.001098


### Load GBIF and map to patches

In [3]:
import rioxarray
from s3_utils import load_zarr
from rasterio.transform import rowcol

# Patch labels raster for coordinate mapping
patch_labels_da = load_zarr(
    config.S3_PROCESSED + '/patches/patch_labels_30m.zarr',
    name='patch_label',
)
patch_labels = patch_labels_da.values
transform = patch_labels_da.rio.transform()

# GBIF
gbif = pd.read_parquet(config.GBIF_PARQUET, storage_options={'anon': False})

# Vectorized mapping: all records → patch IDs
rows, cols = rowcol(transform, gbif['decimallongitude'].values,
                    gbif['decimallatitude'].values)
rows, cols = np.array(rows), np.array(cols)
h, w = patch_labels.shape
valid = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
gbif_patch_ids = np.full(len(gbif), -1, dtype=np.int32)
gbif_patch_ids[valid] = patch_labels[rows[valid], cols[valid]]
gbif['patch_id'] = gbif_patch_ids

# Group by species
MIN_RECORDS = 100
species_counts = gbif['species'].value_counts()
species_patch_groups = (
    gbif[gbif['patch_id'] >= 0]
    .groupby('species')['patch_id']
    .unique()
)
species_patches = {
    sp: set(patches)
    for sp, patches in species_patch_groups.items()
    if species_counts.get(sp, 0) >= MIN_RECORDS
}

print(f'Species with >= {MIN_RECORDS} records: {len(species_patches):,}')

# Free the raster
del patch_labels, patch_labels_da
import gc; gc.collect()


Species with >= 100 records: 3,756


10515

### GNN-SDM Model (flexible architecture)

Configurable number of GraphSAGE layers and hidden dimensions.
The paper used [24, 18, 8] but we'll test alternatives for flora.

In [4]:
class GNNSDM(torch.nn.Module):
    def __init__(self, in_channels, hidden_dims=[24, 18, 8], dropout=0.2):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.dropout = dropout

        prev = in_channels
        for dim in hidden_dims:
            self.convs.append(SAGEConv(prev, dim, aggr='mean'))
            prev = dim

        self.out = torch.nn.Linear(prev, 1)

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = F.leaky_relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return torch.sigmoid(self.out(x)).squeeze(-1)

# Test
m = GNNSDM(n_features, [24, 18, 8])
print(m)
print(f'Parameters: {sum(p.numel() for p in m.parameters()):,}')


GNNSDM(
  (convs): ModuleList(
    (0): SAGEConv(12, 24, aggr=mean)
    (1): SAGEConv(24, 18, aggr=mean)
    (2): SAGEConv(18, 8, aggr=mean)
  )
  (out): Linear(in_features=8, out_features=1, bias=True)
)
Parameters: 1,787


### Training function

Per-species training: label presence patches, select background via
One-Class SVM, weight presence by PageRank, train with MSE loss.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score


def train_gnn_species(species_name, presence_patches, X, edge_index,
                      pr_values, n_patches, device,
                      hidden_dims=[24, 18, 8], epochs=500, lr=0.001,
                      patience=50, return_history=False):
    """
    Train GNN-SDM for one species with early stopping.
    Stops if val AUC doesn't improve for `patience` epochs.
    Returns: (suitability, best_val_auc) or (suitability, best_val_auc, history)
    """
    presence = np.array(list(presence_patches))
    n_pres = len(presence)

    X_np = X.cpu().numpy()
    oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1)
    oc_svm.fit(X_np[presence])

    non_presence = np.setdiff1d(np.arange(n_patches), presence)
    scores = oc_svm.decision_function(X_np[non_presence])
    n_bg = min(len(non_presence), n_pres * 2)
    bg_idx = non_presence[np.argsort(scores)[:n_bg]]

    labels = torch.zeros(n_patches, dtype=torch.float32)
    labels[presence] = 1.0
    weights = torch.zeros(n_patches, dtype=torch.float32)
    weights[presence] = pr_values[presence]
    weights[presence] = weights[presence] / weights[presence].sum() * n_pres
    weights[bg_idx] = 1.0

    labelled = np.concatenate([presence, bg_idx])
    rng = np.random.default_rng(42)
    rng.shuffle(labelled)
    split = int(0.8 * len(labelled))
    train_mask = torch.zeros(n_patches, dtype=torch.bool, device=device)
    val_mask = torch.zeros(n_patches, dtype=torch.bool, device=device)
    train_mask[labelled[:split]] = True
    val_mask[labelled[split:]] = True

    labels = labels.to(device)
    weights = weights.to(device)

    model = GNNSDM(X.shape[1], hidden_dims).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_auc = 0.0
    best_state = None
    epochs_no_improve = 0
    history = [] if return_history else None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X, edge_index)
        loss = (weights[train_mask] * (pred[train_mask] - labels[train_mask]) ** 2).mean()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                pred_all = model(X, edge_index)
                val_pred = pred_all[val_mask].cpu().numpy()
                val_labels = labels[val_mask].cpu().numpy()
                train_loss = loss.item()

            val_auc = 0.0
            if len(np.unique(val_labels)) == 2:
                val_auc = roc_auc_score(val_labels, val_pred)
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 10

            if return_history:
                val_loss = (weights[val_mask] * (pred_all[val_mask] - labels[val_mask]) ** 2).mean().item()
                history.append({'epoch': epoch + 1, 'train_loss': train_loss,
                                'val_loss': val_loss, 'val_auc': val_auc})

            # Early stopping
            if epochs_no_improve >= patience:
                break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        suitability = model(X, edge_index).cpu().numpy()

    if return_history:
        return suitability, best_val_auc, history
    return suitability, best_val_auc

print('Training function defined (with early stopping).')


### Set architecture from notebook 20 results

In [ ]:
# Set this based on the architecture search results
best_hidden_dims = [64, 48, 32]  # adjust after running notebook 20
EPOCHS = 500
PATIENCE = 50

print(f'Architecture: {best_hidden_dims}')
print(f'Max epochs: {EPOCHS}, patience: {PATIENCE}')


### Train for all species

In [6]:
import os
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, cohen_kappa_score,
)

CHECKPOINT_FILE = 'gnn_sdm_checkpoint.csv'
CHECKPOINT_EVERY = 50
EPOCHS = 200

# Move shared tensors to GPU once
X_gpu = X.to(device)
edge_index_gpu = edge_index.to(device)

# Resume
if os.path.exists(CHECKPOINT_FILE):
    done_df = pd.read_csv(CHECKPOINT_FILE)
    results = done_df.to_dict('records')
    done_species = set(done_df['species'])
    print(f'Resuming: {len(done_species)} done')
else:
    results = []
    done_species = set()

species_list = [sp for sp in species_patches if sp not in done_species]
print(f'Training GNN-SDM for {len(species_list)} species ({EPOCHS} epochs each)...')
t0 = time.time()

for i, sp in enumerate(species_list):
    presence = species_patches[sp]
    if len(presence) < 5:
        continue

    suitability, val_auc = train_gnn_species(
        sp, presence, X_gpu, edge_index_gpu,
        pr_values, n_patches, device, epochs=EPOCHS,
    )

    # Evaluate on all labelled patches (presence + background)
    pres_arr = np.array(list(presence))
    all_ids = np.arange(n_patches)
    absence = np.setdiff1d(all_ids, pres_arr)
    rng = np.random.default_rng(42)
    abs_sample = rng.choice(absence, min(len(absence), len(pres_arr) * 3), replace=False)

    eval_idx = np.concatenate([pres_arr, abs_sample])
    y_true = np.concatenate([np.ones(len(pres_arr)), np.zeros(len(abs_sample))])
    y_score = suitability[eval_idx]
    y_pred = (y_score >= 0.5).astype(int)

    results.append({
        'species': sp,
        'n_records': int(species_counts[sp]),
        'n_presence_patches': len(pres_arr),
        'val_auc': val_auc,
        'auc_mean': roc_auc_score(y_true, y_score),
        'accuracy_mean': accuracy_score(y_true, y_pred),
        'precision_mean': precision_score(y_true, y_pred, zero_division=0),
        'recall_mean': recall_score(y_true, y_pred),
        'f1_mean': f1_score(y_true, y_pred),
        'mcc_mean': matthews_corrcoef(y_true, y_pred),
        'kappa_mean': cohen_kappa_score(y_true, y_pred),
        'tss_mean': recall_score(y_true, y_pred) + recall_score(y_true, y_pred, pos_label=0) - 1,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(species_list) - i - 1)
        print(f'  {i+1}/{len(species_list)}  '
              f'elapsed={elapsed:.0f}s  ETA={eta:.0f}s  '
              f'last AUC={results[-1]["auc_mean"]:.3f}')

pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
elapsed = time.time() - t0
print(f'\nDone: {len(results)} species in {elapsed:.0f}s ({elapsed/60:.1f} min)')


Training GNN-SDM for 3756 species (200 epochs each)...
  50/3756  elapsed=222s  ETA=16475s  last AUC=0.698
  100/3756  elapsed=418s  ETA=15275s  last AUC=0.841
  150/3756  elapsed=621s  ETA=14936s  last AUC=0.758


KeyboardInterrupt: 

### Results and comparison with RF baseline

In [ ]:
import matplotlib.pyplot as plt

gnn_df = pd.DataFrame(results)

# Load RF baseline
rf_df = pd.read_csv('baseline_rf_results.csv')

# Merge on species
comp = gnn_df.merge(rf_df, on='species', suffixes=('_gnn', '_rf'))
print(f'Species in both: {len(comp)}')

metrics = ['auc', 'accuracy', 'precision', 'recall', 'f1', 'mcc', 'kappa', 'tss']

print('\n--- Mean metrics ---')
print(f'{"Metric":12s}  {"RF":>8s}  {"GNN":>8s}  {"Δ":>8s}')
print('-' * 40)
for m in metrics:
    rf_col = f'{m}_mean_rf'
    gnn_col = f'{m}_mean_gnn'
    if rf_col in comp.columns and gnn_col in comp.columns:
        rf_val = comp[rf_col].mean()
        gnn_val = comp[gnn_col].mean()
        print(f'{m:12s}  {rf_val:8.3f}  {gnn_val:8.3f}  {gnn_val - rf_val:+8.3f}')

# Scatter: GNN AUC vs RF AUC
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(comp['auc_mean_rf'], comp['auc_mean_gnn'], alpha=0.3, s=10)
ax.plot([0.5, 1], [0.5, 1], 'r--', label='Equal performance')
ax.set_xlabel('RF AUC')
ax.set_ylabel('GNN-SDM AUC')
ax.set_title(f'GNN-SDM vs RF Baseline ({len(comp)} species)')
ax.legend()
ax.set_xlim(0.5, 1)
ax.set_ylim(0.5, 1)
plt.tight_layout()
plt.show()

# How many species improved?
improved = (comp['auc_mean_gnn'] > comp['auc_mean_rf']).sum()
print(f'\nGNN better than RF: {improved}/{len(comp)} '
      f'({improved/len(comp)*100:.1f}%)')

gnn_df.to_csv('gnn_sdm_results.csv', index=False)
print('Saved gnn_sdm_results.csv')
